# SST Next-Brick Pose Prediction — z-Refactored

z is tightly coupled with layer_id and binary state b — so we **do not** ask the MDN to predict it.

**Formulation:**
```
history
  ↓
Transformer encoder
  ↓
predict b           (binary classification)
  ↓
predict layer_id    (multi-class classification)
  ↓
MDN over [x, y, sin_r, cos_r]   conditioned on (b, layer_id)
  ↓
snap z from layer_id lookup
  ↓
hard validator
```

**Data:** batch2 and batch3 validated_simPhysics sequences

In [ ]:
import json
import os
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

cwd = Path(os.getcwd())
if cwd.name == "training_data":
    os.chdir(cwd.parent)
print(f"Working directory: {os.getcwd()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")

In [ ]:
# Data
BATCH_DIRS = [
    "training_data/batch2/validated_simPhysics",
    "training_data/batch3/validated_simPhysics",
]
SEQ_SUBPATH = "5d_sequence/sequence.json"

# Model
FEATURE_DIM  = 33    # rich token: base(7)+layer_state(3)+sup1(6)+sup2(6)+pair(6)+same_layer(5)
POSE_DIM     = 4     # MDN output: [u/std_uv, v/std_uv, sin_r_rel, cos_r_rel] (support-relative)
HIDDEN_DIM   = 128   # kept small to reduce overfitting
N_HEADS      = 4
N_LAYERS     = 2
FF_DIM       = 256
DROPOUT      = 0.1
K_MIXTURES   = 5
MAX_BRICKS   = 60
MAX_LAYER_NORM = 20  # normalisation divisor for layer_id

# Training
BATCH_SIZE   = 64
LR           = 3e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 150
PATIENCE     = 20
GRAD_CLIP    = 1.0
LAMBDA_B     = 1.0
LAMBDA_LAYER = 1.0
N_AUG        = 50

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Data Loading

In [ ]:
def load_sequences(batch_dirs, seq_subpath=SEQ_SUBPATH):
    sequences = []
    for bd in batch_dirs:
        bd_path = Path(bd)
        if not bd_path.exists():
            print(f"Warning: {bd_path} not found, skipping.")
            continue
        for demo_dir in sorted(bd_path.iterdir()):
            seq_path = demo_dir / seq_subpath
            if seq_path.exists():
                with open(seq_path) as f:
                    poses = json.load(f)
                sequences.append({"id": str(demo_dir), "poses": poses})
    return sequences


sequences = load_sequences(BATCH_DIRS)
print(f"Loaded {len(sequences)} sequences:\n")
for s in sequences:
    print(f"  {s['id']}  ->  {len(s['poses'])} bricks")

## 2. Feature Engineering

In [ ]:
def assign_layer_ids(poses, z_tol=1e-4):
    """Group bricks into layers by clustering similar z values."""
    unique_z = []
    for p in poses:
        z = p[2]
        if not any(abs(z - uz) < z_tol for uz in unique_z):
            unique_z.append(z)
    unique_z.sort()
    return [
        min(range(len(unique_z)), key=lambda i: abs(unique_z[i] - p[2]))
        for p in poses
    ]


def canonicalize_r(r):
    """Map yaw r to [0, pi) exploiting brick 180-degree symmetry."""
    r = r % math.pi
    return r + math.pi if r < 0 else r


# ── Support-frame geometry ────────────────────────────────────────────────────

def get_support_frame(target_xy, context_poses, context_layer_ids, target_layer_id):
    """
    Build a 2-D reference frame from the nearest lower-layer bricks.

    target_xy         : [x, y] of the target (or proxy) brick
    context_poses     : list of [x, y, z, b, r]
    context_layer_ids : matching list of int
    target_layer_id   : layer_id of the brick being placed

    Returns dict:
      origin       : [ox, oy]
      x_axis       : [ax, ay]  unit vector
      y_axis       : [bx, by]  perpendicular unit vector
      axis_angle   : float
      has_support_1: bool
      has_support_2: bool  (True → pair frame)
      is_first_layer: bool
    """
    tx, ty = target_xy
    below  = target_layer_id - 1
    is_first_layer = (target_layer_id == 0 or below < 0)

    sup = sorted(
        [(context_poses[i], context_layer_ids[i])
         for i in range(len(context_poses))
         if context_layer_ids[i] == below],
        key=lambda ps: math.sqrt((ps[0][0] - tx) ** 2 + (ps[0][1] - ty) ** 2),
    )

    if is_first_layer or len(sup) == 0:
        return dict(origin=[0.0, 0.0], x_axis=[1.0, 0.0], y_axis=[0.0, 1.0],
                    axis_angle=0.0,
                    has_support_1=False, has_support_2=False, is_first_layer=True)

    if len(sup) == 1:
        s1  = sup[0][0]
        r_c = canonicalize_r(s1[4])
        c, s = math.cos(r_c), math.sin(r_c)
        return dict(origin=[s1[0], s1[1]], x_axis=[c, s], y_axis=[-s, c],
                    axis_angle=r_c,
                    has_support_1=True, has_support_2=False, is_first_layer=False)

    s1, s2 = sup[0][0], sup[1][0]
    mid_x, mid_y = (s1[0] + s2[0]) / 2.0, (s1[1] + s2[1]) / 2.0
    pdx, pdy = s2[0] - s1[0], s2[1] - s1[1]
    d = math.sqrt(pdx * pdx + pdy * pdy + 1e-8)
    ax, ay = pdx / d, pdy / d
    return dict(origin=[mid_x, mid_y], x_axis=[ax, ay], y_axis=[-ay, ax],
                axis_angle=math.atan2(ay, ax),
                has_support_1=True, has_support_2=True, is_first_layer=False)


def world_to_support_frame(tx, ty, tr, sf, std_uv):
    """Convert world-space pose to support-frame-relative coords."""
    ox, oy = sf["origin"]
    ax, ay = sf["x_axis"]
    bx, by = sf["y_axis"]
    dx, dy = tx - ox, ty - oy
    u = ax * dx + ay * dy
    v = bx * dx + by * dy
    r_rel = canonicalize_r(canonicalize_r(tr) - sf["axis_angle"])
    return [u / std_uv, v / std_uv, math.sin(r_rel), math.cos(r_rel)]


def support_frame_to_world(u_norm, v_norm, sin_r_rel, cos_r_rel, sf, std_uv):
    """Decode support-frame-relative MDN prediction to world pose (x, y, r)."""
    ox, oy = sf["origin"]
    ax, ay = sf["x_axis"]
    bx, by = sf["y_axis"]
    u, v = u_norm * std_uv, v_norm * std_uv
    x = ox + ax * u + bx * v
    y = oy + ay * u + by * v
    nrm   = math.sqrt(sin_r_rel ** 2 + cos_r_rel ** 2 + 1e-8)
    r_world = canonicalize_r(sf["axis_angle"]
                             + math.atan2(sin_r_rel / nrm, cos_r_rel / nrm))
    return x, y, r_world


# ── Rich 33-dim brick encoder ─────────────────────────────────────────────────

def encode_brick_rich(pose_5d, layer_id, time_index,
                      context_poses, context_layer_ids,
                      norm_stats, max_layer=MAX_LAYER_NORM):
    """
    33-dim rich feature vector for one brick (Transformer input token).

    Feature layout:
      base pose      (7):  x_n, y_n, b, sin_r, cos_r, layer_norm, time_norm
      layer state    (3):  rel_to_top, is_top, is_second_top
      support 1      (6):  has_s1, dx, dy, sin_dr, cos_dr, dist
      support 2      (6):  has_s2, dx, dy, sin_dr, cos_dr, dist
      support pair   (6):  has_pair, pair_dist, pair_ax_sin, pair_ax_cos, u, v
      same-layer occ (5):  count_norm, ndx, ndy, ndist, is_frontier
    """
    ns = norm_stats
    x, y, z, b, r = pose_5d
    r_c = canonicalize_r(r)
    sin_r, cos_r = math.sin(r_c), math.cos(r_c)
    max_lid  = max(context_layer_ids, default=layer_id)
    max_lid  = max(max_lid, layer_id)
    std_dist = (ns["std_x"] + ns["std_y"]) / 2.0

    feat = [
        (x - ns["mean_x"]) / ns["std_x"],
        (y - ns["mean_y"]) / ns["std_y"],
        float(b), sin_r, cos_r,
        layer_id / max_layer,
        time_index / 60.0,
    ]
    feat += [
        (max_lid - layer_id) / max(max_lid, 1),
        float(layer_id == max_lid),
        float(layer_id == max_lid - 1),
    ]

    below    = layer_id - 1
    supports = sorted(
        [(context_poses[i], context_layer_ids[i])
         for i in range(len(context_poses))
         if context_layer_ids[i] == below],
        key=lambda ps: math.sqrt((ps[0][0] - x) ** 2 + (ps[0][1] - y) ** 2),
    )

    def _sup_feat(sp):
        sx, sy, _sz, _sb, sr = sp
        dx, dy = sx - x, sy - y
        dist   = math.sqrt(dx * dx + dy * dy + 1e-8)
        dr     = canonicalize_r(canonicalize_r(sr) - r_c)
        return [1.0, dx / ns["std_x"], dy / ns["std_y"],
                math.sin(dr), math.cos(dr), dist / std_dist]

    _no_sup = [0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
    feat += _sup_feat(supports[0][0]) if len(supports) >= 1 else _no_sup
    feat += _sup_feat(supports[1][0]) if len(supports) >= 2 else _no_sup

    if len(supports) >= 2:
        s1x, s1y = supports[0][0][0], supports[0][0][1]
        s2x, s2y = supports[1][0][0], supports[1][0][1]
        mid_x, mid_y = (s1x + s2x) / 2.0, (s1y + s2y) / 2.0
        pdx, pdy  = s2x - s1x, s2y - s1y
        pair_dist = math.sqrt(pdx * pdx + pdy * pdy + 1e-8)
        pax_sin, pax_cos = pdy / pair_dist, pdx / pair_dist
        fmx, fmy  = x - mid_x, y - mid_y
        u =  pax_cos * fmx + pax_sin * fmy
        v = -pax_sin * fmx + pax_cos * fmy
        feat += [1.0, pair_dist / std_dist, pax_sin, pax_cos,
                 u / ns["std_x"], v / ns["std_y"]]
    else:
        feat += [0.0, 0.0, 0.0, 1.0, 0.0, 0.0]

    same = [context_poses[i] for i in range(len(context_poses))
            if context_layer_ids[i] == layer_id]
    count_norm = len(same) / 12.0
    if same:
        dxdy = [(p[0]-x, p[1]-y, math.sqrt((p[0]-x)**2+(p[1]-y)**2)) for p in same]
        ndx, ndy, ndist = min(dxdy, key=lambda d: d[2])
        is_frontier = float(len(same) <= 2)
    else:
        ndx, ndy, ndist, is_frontier = 0.0, 0.0, 0.0, 1.0
    feat += [count_norm, ndx / ns["std_x"], ndy / ns["std_y"],
             ndist / std_dist, is_frontier]

    assert len(feat) == 33, f"Expected 33 features, got {len(feat)}"
    return feat

### Compute layer vocabulary size

We look at all loaded sequences to set the classification head output size.  
This is an architectural capacity parameter, not a learned value.

In [ ]:
# MAX_LAYER_CLASSES: maximum layer_id + 1, scanned over all sequences
_max_lid = max(
    lid
    for seq in sequences
    for lid in assign_layer_ids(seq["poses"])
)
MAX_LAYER_CLASSES = _max_lid + 1
print(f"Maximum layer ID seen across all sequences: {_max_lid}")
print(f"Using MAX_LAYER_CLASSES = {MAX_LAYER_CLASSES}")

## 3. Sequence -> Next-Step Training Pairs

Key change: target is now `(x, y, sin_r, cos_r)` + discrete `b` + discrete `layer_id`.  
z is stored for reference but is **not** a prediction target.

In [ ]:
def sequence_to_pairs(seq_data):
    """
    Convert one full sequence to T next-step (history -> target) pairs.

    history_raw stores raw [x,y,z,b,r] poses (not encoded) so that SE(2)
    augmentation can transform them directly.  encode_brick_rich is called
    lazily in BrickPoseDataset.__getitem__ after augmentation.
    """
    poses     = seq_data["poses"]
    layer_ids = assign_layer_ids(poses)
    pairs = []
    for t in range(len(poses)):
        x, y, z, b, r = poses[t]
        r_c = canonicalize_r(r)
        target = {
            "x": x, "y": y,
            "b": int(b),
            "sin_r": math.sin(r_c),
            "cos_r": math.cos(r_c),
            "layer_id": layer_ids[t],
            "z": z,                    # reference only — NOT a prediction target
        }
        pairs.append({
            "history_raw":      [list(poses[i]) for i in range(t)],
            "history_layer_ids": [layer_ids[i]  for i in range(t)],
            "target":            target,
            "seq_id":            seq_data["id"],
        })
    return pairs

## 4. Sequence-Level Train / Val / Test Split

In [ ]:
indices = list(range(len(sequences)))
random.shuffle(indices)

n_total = len(sequences)
n_val   = max(1, round(0.15 * n_total))
n_test  = max(1, round(0.15 * n_total))
n_train = n_total - n_val - n_test

train_idx = indices[:n_train]
val_idx   = indices[n_train : n_train + n_val]
test_idx  = indices[n_train + n_val :]

train_seqs = [sequences[i] for i in train_idx]
val_seqs   = [sequences[i] for i in val_idx]
test_seqs  = [sequences[i] for i in test_idx]

print(f"Split: {n_train} train / {n_val} val / {n_test} test  (total {n_total})")
print(f"Train: {[Path(sequences[i]['id']).name for i in train_idx]}")
print(f"Val:   {[Path(sequences[i]['id']).name for i in val_idx]}")
print(f"Test:  {[Path(sequences[i]['id']).name for i in test_idx]}")

train_pairs_orig = [p for seq in train_seqs for p in sequence_to_pairs(seq)]
val_pairs        = [p for seq in val_seqs   for p in sequence_to_pairs(seq)]
test_pairs       = [p for seq in test_seqs  for p in sequence_to_pairs(seq)]

print(f"\nPairs before aug -- train: {len(train_pairs_orig)}, val: {len(val_pairs)}, test: {len(test_pairs)}")

### Build z-Lookup Table (training data only)

Maps `layer_id -> mean z` so we can snap z after sampling.  
Built from training sequences **only** to respect the data split.

In [ ]:
def build_z_lookup(seqs):
    """Map layer_id -> mean z across given sequences (training only)."""
    acc = {}
    for seq in seqs:
        poses     = seq["poses"]
        layer_ids = assign_layer_ids(poses)
        for p, lid in zip(poses, layer_ids):
            acc.setdefault(lid, []).append(p[2])
    return {lid: float(np.mean(zs)) for lid, zs in sorted(acc.items())}


z_lookup = build_z_lookup(train_seqs)
print("z_lookup (layer_id -> mean z from training data):")
for lid, z_val in z_lookup.items():
    print(f"  layer {lid:2d}: z = {z_val:.6f} m")

# Safe z-snap: if predicted layer_id is out of lookup, use the last known z
def snap_z(layer_id):
    if layer_id in z_lookup:
        return z_lookup[layer_id]
    return z_lookup[max(z_lookup.keys())]  # extrapolate upward

## 5. SE(2) Data Augmentation  (training only)

Augmentation applies only to x, y, and rotation — **z is unchanged**,  
so the layer_id and z reference in the target remain correct after augmentation.

In [ ]:
def _rotate_xy(x, y, cx, cy, cos_t, sin_t, tx, ty):
    dx, dy = x - cx, y - cy
    return cx + cos_t * dx - sin_t * dy + tx, cy + sin_t * dx + cos_t * dy + ty


def se2_augment(pair, theta, tx, ty):
    """
    Apply a rigid SE(2) transform to a raw-pose pair.

    z, layer_id, and binary state b are invariant under this transform.
    history_raw is a list of raw [x, y, z, b, r] poses.
    """
    cos_t, sin_t = math.cos(theta), math.sin(theta)

    all_x = [h[0] for h in pair["history_raw"]] + [pair["target"]["x"]]
    all_y = [h[1] for h in pair["history_raw"]] + [pair["target"]["y"]]
    cx    = sum(all_x) / len(all_x)
    cy    = sum(all_y) / len(all_y)

    new_history_raw = []
    for h in pair["history_raw"]:
        x, y, z, b, r = h
        xn, yn = _rotate_xy(x, y, cx, cy, cos_t, sin_t, tx, ty)
        r_new  = canonicalize_r(r + theta)
        new_history_raw.append([xn, yn, z, b, r_new])

    tgt    = pair["target"]
    xn, yn = _rotate_xy(tgt["x"], tgt["y"], cx, cy, cos_t, sin_t, tx, ty)
    r_c    = math.atan2(tgt["sin_r"], tgt["cos_r"])
    r_new  = canonicalize_r(r_c + theta)
    new_target = {**tgt,
                  "x": xn, "y": yn,
                  "sin_r": math.sin(r_new), "cos_r": math.cos(r_new)}

    return {
        "history_raw":       new_history_raw,
        "history_layer_ids": pair["history_layer_ids"],  # unchanged by SE(2)
        "target":            new_target,
        "seq_id":            pair["seq_id"],
    }


def augment_pairs(pairs, n_aug):
    aug = []
    for pair in pairs:
        for _ in range(n_aug):
            aug.append(se2_augment(
                pair,
                theta=random.uniform(0, 2 * math.pi),
                tx=random.uniform(-0.01, 0.01),
                ty=random.uniform(-0.01, 0.01),
            ))
    return aug


print(f"Generating {N_AUG} augmentations x {len(train_pairs_orig)} training pairs...")
train_pairs_aug = augment_pairs(train_pairs_orig, N_AUG)
all_train_pairs = train_pairs_orig + train_pairs_aug
print(f"Total training pairs after augmentation: {len(all_train_pairs):,}")

## 6. Normalization Statistics  (training data only)

x, y, z normalisation is used for **input tokens** (z still context for the encoder).  
Only x, y normalisation is used for **target denormalisation** at inference.

In [ ]:
def compute_norm_stats(pairs):
    """Collect x, y, z statistics from raw history and target poses."""
    xs, ys, zs = [], [], []
    for p in pairs:
        xs.append(p["target"]["x"])
        ys.append(p["target"]["y"])
        zs.append(p["target"]["z"])
        for h in p["history_raw"]:   # h is [x, y, z, b, r]
            xs.append(h[0]); ys.append(h[1]); zs.append(h[2])
    return {
        "mean_x": float(np.mean(xs)), "std_x": float(np.std(xs)) + 1e-8,
        "mean_y": float(np.mean(ys)), "std_y": float(np.std(ys)) + 1e-8,
        "mean_z": float(np.mean(zs)), "std_z": float(np.std(zs)) + 1e-8,
    }


norm_stats = compute_norm_stats(all_train_pairs)
print("Normalization statistics (from training data only):")
for k, v in norm_stats.items():
    print(f"  {k:8s}: {v:.6f}")

## 7. PyTorch Dataset & DataLoaders

Each sample returns **five** tensors:
```
tokens       (MAX_BRICKS, 33)  -- rich feature tokens; encode_brick_rich called lazily per item
mask         (MAX_BRICKS,)     -- True = real brick
target_cont  (4,)              -- [x_norm, y_norm, sin_r, cos_r]
target_b     scalar            -- 0 (laying) or 1 (standing)
target_layer scalar            -- discrete layer class 0..MAX_LAYER_CLASSES-1
```

Rich features are computed inside `__getitem__` so that SE(2) augmentation can
transform the raw [x, y, z, b, r] poses directly without needing to undo encoding.

In [ ]:
class BrickPoseDataset(Dataset):
    """
    Each sample returns six tensors:
      tokens       (MAX_BRICKS, 33)  -- rich feature tokens
      mask         (MAX_BRICKS,)     -- True = real brick
      target_cont  (4,)              -- [u/std_uv, v/std_uv, sin_r_rel, cos_r_rel]
      target_b     scalar            -- 0 (laying) or 1 (standing)
      target_layer scalar            -- discrete layer class
      target_has_pair scalar (float) -- 1.0 if support pair exists, else 0.0
    """

    def __init__(self, pairs, norm_stats, max_bricks=MAX_BRICKS):
        self.pairs      = pairs
        self.ns         = norm_stats
        self.max_bricks = max_bricks
        self.std_uv     = (norm_stats["std_x"] + norm_stats["std_y"]) / 2.0

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair      = self.pairs[idx]
        hist_raw  = pair["history_raw"]        # list of [x,y,z,b,r]
        hist_lids = pair["history_layer_ids"]  # list of int
        tgt       = pair["target"]
        tgt_layer_id = tgt["layer_id"]
        ns        = self.ns
        std_uv    = self.std_uv

        # ── Input tokens ──────────────────────────────────────────────────────
        tokens = torch.zeros(self.max_bricks, FEATURE_DIM, dtype=torch.float32)
        mask   = torch.zeros(self.max_bricks, dtype=torch.bool)
        for i, (pose, lid) in enumerate(zip(hist_raw[:self.max_bricks],
                                            hist_lids[:self.max_bricks])):
            feat = encode_brick_rich(
                pose, lid, i, hist_raw[:i], hist_lids[:i], ns,
            )
            tokens[i] = torch.tensor(feat, dtype=torch.float32)
            mask[i]   = True

        # ── Support frame for the target brick ────────────────────────────────
        # Use ground-truth target position to find nearest support bricks
        # (teacher forcing: same rule as inference uses the correct frame)
        tgt_r = math.atan2(tgt["sin_r"], tgt["cos_r"])
        sf    = get_support_frame(
            [tgt["x"], tgt["y"]],
            hist_raw, hist_lids, tgt_layer_id,
        )
        sup_rel = world_to_support_frame(tgt["x"], tgt["y"], tgt_r, sf, std_uv)
        target_cont     = torch.tensor(sup_rel, dtype=torch.float32)   # [u_n, v_n, sin, cos]
        target_b        = torch.tensor(tgt["b"],        dtype=torch.long)
        target_layer    = torch.tensor(tgt["layer_id"], dtype=torch.long)
        target_has_pair = torch.tensor(1.0 if sf["has_support_2"] else 0.0,
                                       dtype=torch.float32)

        return tokens, mask, target_cont, target_b, target_layer, target_has_pair


train_ds = BrickPoseDataset(all_train_pairs, norm_stats)
val_ds   = BrickPoseDataset(val_pairs,       norm_stats)
test_ds  = BrickPoseDataset(test_pairs,      norm_stats)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds):,} samples  ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,} samples  ({len(val_loader)} batches)")
print(f"Test:  {len(test_ds):,} samples  ({len(test_loader)} batches)")
print(f"std_uv = {train_ds.std_uv:.4f} m")

## 8. Model Architecture

```
tokens  ->  MLP proj  ->  Transformer (pre-norm)  ->  CLS
                                                         |
                              +---------------------------+---------------------------+
                              |                           |                           |
                         b_head                    layer_head              [scene | b_cond | layer_cond]
                         (binary)              (layer classify)                        |
                                                                               MDN over [x,y,sin_r,cos_r]
```

The MDN head receives `[scene (128D) | b_scalar (1D) | layer_norm (1D)]` = 130D input,  
enabling it to learn different spatial distributions for each (b, layer) combination.

**Teacher forcing during training:** ground-truth b and layer_id are passed as conditioning.  
**Inference:** predicted b and layer_id are used.

In [ ]:
class NextBrickModel(nn.Module):
    """
    Hierarchical model: b -> layer_id -> MDN over [u/std_uv, v/std_uv, sin_r_rel, cos_r_rel].

    Input tokens : 33-dim rich features (no z in input; z is rule-derived).
    MDN input    : [scene (HIDDEN_DIM) | b (1) | layer_norm (1) | has_pair (1)] = HIDDEN_DIM+3.

    Teacher forcing during training:
      ground-truth b, layer_id, and has_support_pair condition the MDN.
    Inference:
      predicted b / layer_id; has_pair from the deterministic scene rule.
    """

    def __init__(
        self,
        feature_dim=FEATURE_DIM, hidden_dim=HIDDEN_DIM,
        nhead=N_HEADS, num_layers=N_LAYERS, ff_dim=FF_DIM,
        dropout=DROPOUT, K=K_MIXTURES,
        max_layer_classes=None,
    ):
        super().__init__()
        self.K          = K
        self.pose_dim   = POSE_DIM
        n_layers_cls    = max_layer_classes or MAX_LAYER_CLASSES

        self.input_proj = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.b_head = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 2),
        )
        self.layer_head = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_layers_cls),
        )

        # MDN conditioned on (b, layer_id_norm, has_support_pair)
        mdn_in = hidden_dim + 3
        self.mdn_pi        = nn.Linear(mdn_in, K)
        self.mdn_mu        = nn.Linear(mdn_in, K * self.pose_dim)
        self.mdn_log_sigma = nn.Linear(mdn_in, K * self.pose_dim)

    def forward(self, tokens, mask,
                cond_b=None, cond_layer=None, cond_has_pair=None):
        """
        cond_b        : (B,) long   ground-truth b  (None → use predicted)
        cond_layer    : (B,) long   ground-truth layer_id  (None → use predicted)
        cond_has_pair : (B,) float  1.0 if support pair exists  (None → assume 0)
        """
        B = tokens.shape[0]
        x   = self.input_proj(tokens)
        cls = self.cls_token.expand(B, 1, -1)
        x   = torch.cat([cls, x], dim=1)
        cls_valid = torch.ones(B, 1, dtype=torch.bool, device=mask.device)
        x     = self.transformer(x, src_key_padding_mask=~torch.cat([cls_valid, mask], dim=1))
        scene = x[:, 0]

        b_logits     = self.b_head(scene)
        layer_logits = self.layer_head(scene)

        b_cond = (cond_b.float() if cond_b is not None
                  else b_logits.argmax(-1).float()).unsqueeze(-1)            # (B, 1)

        layer_cond = ((cond_layer.float() if cond_layer is not None
                       else layer_logits.argmax(-1).float())
                      / MAX_LAYER_CLASSES).unsqueeze(-1)                    # (B, 1)

        has_pair_cond = (cond_has_pair.unsqueeze(-1) if cond_has_pair is not None
                         else torch.zeros(B, 1, device=scene.device))      # (B, 1)

        scene_cond = torch.cat([scene, b_cond, layer_cond, has_pair_cond], dim=-1)
        pi    = F.softmax(self.mdn_pi(scene_cond), dim=-1)
        mu    = self.mdn_mu(scene_cond).view(B, self.K, self.pose_dim)
        sigma = (
            F.softplus(self.mdn_log_sigma(scene_cond)).view(B, self.K, self.pose_dim) + 1e-4
        )
        return b_logits, layer_logits, pi, mu, sigma

## 9. Loss Functions

In [ ]:
_LOG2PI = math.log(2 * math.pi)


def mdn_nll_loss(pi, mu, sigma, target):
    """Gaussian MDN NLL via log-sum-exp.  target:(B,4)."""
    target    = target.unsqueeze(1).expand_as(mu)
    log_gauss = (
        -0.5 * (((target - mu) / sigma).pow(2) + 2 * sigma.log() + _LOG2PI)
    ).sum(-1)
    return -torch.logsumexp(torch.log(pi + 1e-8) + log_gauss, dim=-1).mean()


def compute_loss(b_logits, layer_logits, pi, mu, sigma, target_cont, target_b, target_layer):
    l_mdn   = mdn_nll_loss(pi, mu, sigma, target_cont)
    l_b     = F.cross_entropy(b_logits,     target_b)
    l_layer = F.cross_entropy(layer_logits, target_layer)
    total   = l_mdn + LAMBDA_B * l_b + LAMBDA_LAYER * l_layer
    return total, l_mdn, l_b, l_layer

## 10. Training & Validation Loops

Teacher forcing: ground-truth `b` and `layer_id` are passed to the MDN during **both** train and val.  
This gives a fair measure of how well the MDN learns the conditional pose distribution.

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for tokens, mask, tgt_cont, tgt_b, tgt_layer, tgt_has_pair in loader:
        tokens, mask         = tokens.to(device), mask.to(device)
        tgt_cont, tgt_b      = tgt_cont.to(device), tgt_b.to(device)
        tgt_layer            = tgt_layer.to(device)
        tgt_has_pair         = tgt_has_pair.to(device)
        optimizer.zero_grad()
        out = model(tokens, mask,
                    cond_b=tgt_b, cond_layer=tgt_layer, cond_has_pair=tgt_has_pair)
        loss, _, _, _ = compute_loss(*out, tgt_cont, tgt_b, tgt_layer)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    tot, tot_mdn, tot_b, tot_layer = 0.0, 0.0, 0.0, 0.0
    corr_b, corr_layer, n = 0, 0, 0
    for tokens, mask, tgt_cont, tgt_b, tgt_layer, tgt_has_pair in loader:
        tokens, mask     = tokens.to(device), mask.to(device)
        tgt_cont, tgt_b  = tgt_cont.to(device), tgt_b.to(device)
        tgt_layer        = tgt_layer.to(device)
        tgt_has_pair     = tgt_has_pair.to(device)
        out = model(tokens, mask,
                    cond_b=tgt_b, cond_layer=tgt_layer, cond_has_pair=tgt_has_pair)
        b_logits, layer_logits, pi, mu, sigma = out
        loss, l_mdn, l_b, l_layer = compute_loss(*out, tgt_cont, tgt_b, tgt_layer)
        tot += loss.item(); tot_mdn += l_mdn.item()
        tot_b += l_b.item(); tot_layer += l_layer.item()
        corr_b     += (b_logits.argmax(-1)     == tgt_b).sum().item()
        corr_layer += (layer_logits.argmax(-1) == tgt_layer).sum().item()
        n += len(tgt_b)
    N = len(loader)
    return tot/N, tot_mdn/N, tot_b/N, tot_layer/N, corr_b/n, corr_layer/n

## 11. Main Training Loop

In [ ]:
model     = NextBrickModel(
    hidden_dim=HIDDEN_DIM, ff_dim=FF_DIM,
    max_layer_classes=MAX_LAYER_CLASSES,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(f"FEATURE_DIM={FEATURE_DIM}  HIDDEN_DIM={HIDDEN_DIM}  FF_DIM={FF_DIM}")
print(f"MDN output: {POSE_DIM}D [u/std_uv, v/std_uv, sin_r_rel, cos_r_rel]  (support-relative)")

best_val, no_improve = float("inf"), 0
CKPT = Path("training_data/best_model_z_refactored.pth")
log  = {"train": [], "val": [], "val_mdn": [], "val_b": [], "val_layer": [],
        "b_acc": [], "layer_acc": []}

std_uv = train_ds.std_uv   # saved in checkpoint for consistent inference decoding

for epoch in range(1, MAX_EPOCHS + 1):
    tr = train_epoch(model, train_loader, optimizer)
    vl, vl_mdn, vl_b, vl_layer, b_acc, layer_acc = eval_epoch(model, val_loader)
    scheduler.step(vl)

    log["train"].append(tr);           log["val"].append(vl)
    log["val_mdn"].append(vl_mdn);     log["val_b"].append(vl_b)
    log["val_layer"].append(vl_layer); log["b_acc"].append(b_acc)
    log["layer_acc"].append(layer_acc)

    if vl < best_val:
        best_val, no_improve = vl, 0
        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "norm_stats": norm_stats,
            "z_lookup": z_lookup,
            "max_layer_classes": MAX_LAYER_CLASSES,
            "feature_dim": FEATURE_DIM,
            "hidden_dim": HIDDEN_DIM,
            "std_uv": std_uv,    # needed to denormalize u, v at inference
            "val_loss": vl,
        }, CKPT)
    else:
        no_improve += 1

    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"[{epoch:3d}] train={tr:.3f} | val={vl:.3f} "
            f"(mdn={vl_mdn:.3f} b={vl_b:.3f} layer={vl_layer:.3f}) "
            f"| b_acc={b_acc:.3f} layer_acc={layer_acc:.3f} | lr={lr_now:.1e}"
        )
    if no_improve >= PATIENCE:
        print(f"\nEarly stop at epoch {epoch}.")
        break

print(f"\nBest val loss: {best_val:.4f}  ->  {CKPT}")

## 12. Training Curves

In [ ]:
E = range(1, len(log["train"]) + 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].plot(E, log["train"], label="train"); axes[0,0].plot(E, log["val"], label="val")
axes[0,0].set_title("Total Loss"); axes[0,0].set_xlabel("Epoch"); axes[0,0].legend()

axes[0,1].plot(E, log["val_mdn"], label="MDN NLL")
axes[0,1].plot(E, log["val_b"],   label="Binary CE")
axes[0,1].plot(E, log["val_layer"], label="Layer CE")
axes[0,1].set_title("Val Loss Components"); axes[0,1].set_xlabel("Epoch"); axes[0,1].legend()

axes[1,0].plot(E, log["b_acc"])
axes[1,0].set_title("Val Binary-State Accuracy (b)")
axes[1,0].set_xlabel("Epoch"); axes[1,0].set_ylim(0, 1)
axes[1,0].axhline(0.5, color="gray", linestyle="--", alpha=0.5)

axes[1,1].plot(E, log["layer_acc"])
axes[1,1].set_title("Val Layer-ID Accuracy")
axes[1,1].set_xlabel("Epoch"); axes[1,1].set_ylim(0, 1)
axes[1,1].axhline(1/MAX_LAYER_CLASSES, color="gray", linestyle="--", alpha=0.5, label="chance")
axes[1,1].legend()

plt.tight_layout()
plt.savefig("training_data/training_curves_z_refactored.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_data/training_curves_z_refactored.png")

## 13. Inference -- Candidate Sampling with z-Snapping

**Pipeline:**
1. Encode history
2. Model predicts: `b` (discrete) + `layer_id` (discrete)
3. Model samples: `[x, y, sin_r, cos_r]` from MDN conditioned on predicted `(b, layer_id)`
4. Snap z from `z_lookup[layer_id]`
5. Assemble full 5D pose candidate `[x, y, z, b, r]`

In [ ]:
_LOG2PI_NB = math.log(2 * math.pi)


def sample_next_brick(model, history_5d, norm_stats, z_lookup, n_candidates=100):
    """
    Sample n_candidates next brick poses using the z-refactored support-relative model.

    history_5d : list of accepted raw [x, y, z, b, r] poses
    Returns (candidates, b_prob, layer_prob) where candidates are dicts
      {x, y, z, b, sin_r, cos_r, r, layer_id, log_prob}
    """
    ns     = norm_stats
    std_uv = (ns["std_x"] + ns["std_y"]) / 2.0
    model.eval()

    # Build rich-feature tokens from full history
    if history_5d:
        layer_ids = assign_layer_ids(history_5d)
        encoded = [
            encode_brick_rich(history_5d[t], layer_ids[t], t,
                              history_5d[:t], layer_ids[:t], ns)
            for t in range(len(history_5d))
        ]
    else:
        layer_ids = []
        encoded   = []

    tokens = torch.zeros(1, MAX_BRICKS, FEATURE_DIM)
    mask   = torch.zeros(1, MAX_BRICKS, dtype=torch.bool)
    for i, h in enumerate(encoded[:MAX_BRICKS]):
        tokens[0, i] = torch.tensor(h, dtype=torch.float32)
        mask[0, i]   = True

    # First pass: get discrete b and layer predictions
    with torch.no_grad():
        b_logits, layer_logits, _, _, _ = model(tokens.to(device), mask.to(device))

    b_prob     = F.softmax(b_logits[0],     dim=-1).cpu().numpy()
    layer_prob = F.softmax(layer_logits[0], dim=-1).cpu().numpy()
    pred_layer = int(layer_prob.argmax())
    pred_b     = int(b_prob.argmax())
    z_val      = snap_z(pred_layer)

    # Build support frame for this predicted layer (deterministic, Option A)
    below = pred_layer - 1
    if pred_layer > 0 and history_5d:
        sup_bricks = [history_5d[i] for i, lid in enumerate(layer_ids) if lid == below]
        if sup_bricks:
            cx = sum(p[0] for p in sup_bricks) / len(sup_bricks)
            cy = sum(p[1] for p in sup_bricks) / len(sup_bricks)
        else:
            cx = cy = 0.0
        sf = get_support_frame([cx, cy], history_5d, layer_ids, pred_layer)
    else:
        sf = dict(origin=[0.0, 0.0], x_axis=[1.0, 0.0], y_axis=[0.0, 1.0],
                  axis_angle=0.0, has_support_1=False, has_support_2=False,
                  is_first_layer=True)

    has_pair = 1.0 if sf["has_support_2"] else 0.0

    # Second pass: sample MDN conditioned on predicted b, layer, has_pair
    has_pair_t = torch.tensor([has_pair], dtype=torch.float32).to(device)
    b_t        = torch.tensor([pred_b],   dtype=torch.long).to(device)
    layer_t    = torch.tensor([pred_layer], dtype=torch.long).to(device)

    with torch.no_grad():
        _, _, pi, mu, sigma = model(tokens.to(device), mask.to(device),
                                    cond_b=b_t, cond_layer=layer_t,
                                    cond_has_pair=has_pair_t)

    pi_np  = pi[0].cpu().numpy()
    mu_np  = mu[0].cpu().numpy()    # (K, 4): [u_n, v_n, sin_r_rel, cos_r_rel]
    sig_np = sigma[0].cpu().numpy()

    candidates = []
    for _ in range(n_candidates):
        k = np.random.choice(len(pi_np), p=pi_np / pi_np.sum())
        s = np.random.randn(4) * sig_np[k] + mu_np[k]

        u_norm, v_norm       = float(s[0]), float(s[1])
        sin_r_rel, cos_r_rel = float(s[2]), float(s[3])

        # Decode support-relative → world
        xr, yr, rr = support_frame_to_world(u_norm, v_norm, sin_r_rel, cos_r_rel,
                                             sf, std_uv)
        sin_r = math.sin(rr); cos_r = math.cos(rr)

        # Score under mixture in normalised output space
        norm_s    = np.array([u_norm, v_norm, sin_r_rel, cos_r_rel])
        log_gauss = (
            -0.5 * (((norm_s - mu_np) / sig_np) ** 2
                    + 2 * np.log(sig_np) + _LOG2PI_NB)
        ).sum(axis=-1)
        log_prob = float(np.logaddexp.reduce(np.log(pi_np + 1e-8) + log_gauss))

        candidates.append({
            "x": xr, "y": yr, "z": z_val,
            "b": pred_b, "sin_r": sin_r, "cos_r": cos_r, "r": rr,
            "layer_id": pred_layer, "log_prob": log_prob,
        })

    candidates.sort(key=lambda c: c["log_prob"], reverse=True)
    return candidates, b_prob, layer_prob

In [ ]:
# Load best checkpoint
ckpt = torch.load(CKPT, map_location=device)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded checkpoint: epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

# Pick a val pair with a long history for a meaningful demo
demo_pair = max(val_pairs, key=lambda p: len(p["history_raw"]))
print(f"Demo seq: {Path(demo_pair['seq_id']).name}  |  "
      f"history: {len(demo_pair['history_raw'])} bricks")

# Reconstruct 5D history from raw poses for the new API
demo_history_5d = [list(h) for h in demo_pair["history_raw"]]

candidates, b_prob, layer_prob = sample_next_brick(
    model, demo_history_5d, norm_stats, z_lookup, n_candidates=100
)
pred_layer = candidates[0]["layer_id"]
pred_z     = snap_z(pred_layer)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
hx = [h[0] for h in demo_pair["history_raw"]]
hy = [h[1] for h in demo_pair["history_raw"]]
ax.scatter(hx, hy, c="steelblue", s=40, alpha=0.7, label="History", zorder=3)
tgt = demo_pair["target"]
ax.scatter(tgt["x"], tgt["y"], c="limegreen", s=250, marker="*",
           label=f"GT (layer={tgt['layer_id']})", zorder=5)
cx_list = [c["x"] for c in candidates]
cy_list = [c["y"] for c in candidates]
ax.scatter(cx_list, cy_list, c="tomato", s=20, alpha=0.5,
           label=f"Candidates (layer={pred_layer})", zorder=4)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_title("Sampled Candidates — Top-Down View")
ax.legend(); ax.set_aspect("equal")

ax2 = axes[1]
ax2.bar(range(len(layer_prob)), layer_prob, color="steelblue", alpha=0.7)
ax2.axvline(tgt["layer_id"], color="limegreen", linewidth=2,
            label=f"GT layer={tgt['layer_id']}")
ax2.axvline(pred_layer, color="tomato", linewidth=2, linestyle="--",
            label=f"Pred layer={pred_layer} (z={pred_z:.4f}m)")
ax2.set_xlabel("Layer ID"); ax2.set_ylabel("P(layer)")
ax2.set_title("Layer ID Distribution"); ax2.legend()

plt.tight_layout()
plt.savefig("training_data/candidate_visualization_z_refactored.png",
            dpi=150, bbox_inches="tight")
plt.show()

print(f"\nBinary probs  --  laying: {b_prob[0]:.3f}  standing: {b_prob[1]:.3f}")
print(f"Ground truth: b={'laying' if tgt['b']==0 else 'standing'}, "
      f"layer={tgt['layer_id']}, z={tgt['z']:.4f}m")
print(f"Predicted:    layer={pred_layer}, snapped z={pred_z:.4f}m")